In [5]:
!apt-get -qq install -y libopenmpi-dev openmpi-bin > /dev/null
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --allow-run-as-root --version | head -n 2
!pip install -q mpi4py numba numba-cuda numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-77f035de-368b-5dff-3671-c20d66d19e91, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [6]:
GITHUB_USERNAME = "230103010-wq"
assert GITHUB_USERNAME != "Nura" and len(GITHUB_USERNAME.strip()) > 0, ()
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: 230103010-wq


In [17]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import time
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# Compile-time constants
TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2   # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15

# Local subdomain geometry (2x2 process grid)
NY_LOC = GLOBAL_NY // 2     # 256
NX_LOC = GLOBAL_NX // 2     # 256
DX = DY = 1.0 / (GLOBAL_NX + 1)   # 1/513


def get_gpu_uuid() -> str:
    """Query active NVIDIA GPU UUID via nvidia-smi."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Deterministic 64-bit seed bound to GitHub handle and GPU UUID."""
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf8")).hexdigest()
    return int(digest[:16], 16)


# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    world = MPI.COMM_WORLD
    if world.Get_size() != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {world.Get_size()}.")

    cart_comm = world.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    rank = cart_comm.Get_rank()
    row_rank, col_rank = cart_comm.Get_coords(rank)

    nbr_up, nbr_down = cart_comm.Shift(0, 1)      # axis 0 = rows
    nbr_left, nbr_right = cart_comm.Shift(1, 1)   # axis 1 = columns

    # Bind process to GPU 0
    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = cart_comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    return {
        "cart_comm": cart_comm, "rank": rank,
        "row_rank": row_rank, "col_rank": col_rank,
        "nbr_up": nbr_up, "nbr_down": nbr_down,
        "nbr_left": nbr_left, "nbr_right": nbr_right,
        "gpu_uuid": gpu_uuid, "instance_seed": instance_seed,
    }


# ==============================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization
# ==============================================================================
def initialize_subdomains(ctx):
    row_rank, col_rank = ctx["row_rank"], ctx["col_rank"]

    # Global node indices 0..513 (0 and 513 are physical boundaries)
    gi = row_rank * NY_LOC + np.arange(NY_LOC + 2)
    gj = col_rank * NX_LOC + np.arange(NX_LOC + 2)
    y = (gi * DY)[:, None]   # axis 0 -> y
    x = (gj * DX)[None, :]   # axis 1 -> x

    # Diffusivity kappa in [0.08, 0.24]
    kappa = 0.16 + 0.08 * np.sin(2 * np.pi * x) * np.cos(2 * np.pi * y)

    # Gaussian pulses: centers/amplitudes from the seed (identical on all ranks)
    rng = np.random.default_rng(ctx["instance_seed"])
    n_pulses = 3
    cx = rng.uniform(0.2, 0.8, n_pulses)
    cy = rng.uniform(0.2, 0.8, n_pulses)
    amp = rng.uniform(0.5, 1.5, n_pulses)
    sigma = 0.08

    u = np.zeros((NY_LOC + 2, NX_LOC + 2), dtype=np.float64)
    for k in range(n_pulses):
        u += amp[k] * np.exp(-((x - cx[k]) ** 2 + (y - cy[k]) ** 2) / (2 * sigma ** 2))

    # Localized source
    S = 0.05 * np.exp(-((x - 0.5) ** 2 + (y - 0.5) ** 2) / (2 * 0.05 ** 2))

    # Dirichlet u = 0 on outer global edges
    if ctx["nbr_up"] == MPI.PROC_NULL:    u[0, :] = 0.0
    if ctx["nbr_down"] == MPI.PROC_NULL:  u[-1, :] = 0.0
    if ctx["nbr_left"] == MPI.PROC_NULL:  u[:, 0] = 0.0
    if ctx["nbr_right"] == MPI.PROC_NULL: u[:, -1] = 0.0

    kappa_max_local = float(kappa.max())
    return (np.ascontiguousarray(u, dtype=np.float64),
            np.ascontiguousarray(kappa, dtype=np.float64),
            np.ascontiguousarray(S, dtype=np.float64),
            kappa_max_local)
    import time

# ==============================================================================
# TASK 3: GPU halo pack/unpack kernels + pinned async MPI exchange
# ==============================================================================
# Buffer layout (both send and recv), 4 segments of length N = 256:
#   send: 0 = row 1 (up), 1 = row ny (down), 2 = col 1 (left), 3 = col nx (right)
#   recv: 0 = ghost row 0, 1 = ghost row ny+1, 2 = ghost col 0, 3 = ghost col nx+1
HALO_N = 256
# Tag = direction of travel of the message
TAG_UP, TAG_DOWN, TAG_LEFT, TAG_RIGHT = 0, 1, 2, 3


@cuda.jit
def pack_halos_kernel(u, send, ny, nx, N):
    t = cuda.grid(1)
    if t >= 4 * N:
        return
    seg = t // N
    k = t - seg * N
    if seg == 0:
        send[t] = u[1, k + 1]
    elif seg == 1:
        send[t] = u[ny, k + 1]
    elif seg == 2:
        send[t] = u[k + 1, 1]       # strided column read
    else:
        send[t] = u[k + 1, nx]      # strided column read


@cuda.jit
def unpack_halos_kernel(u, recv, ny, nx, N):
    t = cuda.grid(1)
    if t >= 4 * N:
        return
    seg = t // N
    k = t - seg * N
    if seg == 0:
        u[0, k + 1] = recv[t]
    elif seg == 1:
        u[ny + 1, k + 1] = recv[t]
    elif seg == 2:
        u[k + 1, 0] = recv[t]
    else:
        u[k + 1, nx + 1] = recv[t]


class SolverState:
    """Rank-local device/host buffers (only (258,258) arrays, no global array)."""
    def __init__(self, u0, kappa, S, dt):
        self.dt = dt
        self.d_u = cuda.to_device(u0)
        self.d_un = cuda.to_device(u0.copy())
        self.d_kappa = cuda.to_device(kappa)
        self.d_S = cuda.to_device(S)
        self.d_send = cuda.device_array(4 * HALO_N, dtype=np.float64)
        self.d_recv = cuda.device_array(4 * HALO_N, dtype=np.float64)
        self.h_send = cuda.pinned_array(4 * HALO_N, dtype=np.float64)
        self.h_recv = cuda.pinned_array(4 * HALO_N, dtype=np.float64)
        self.h_send[:] = 0.0
        self.h_recv[:] = 0.0   # PROC_NULL segments stay 0.0 -> Dirichlet ghosts
        self.d_send.copy_to_device(self.h_send)
        self.d_recv.copy_to_device(self.h_recv)
        self.stream_boundary = cuda.stream()
        self.stream_interior = cuda.stream()


HALO_BLOCKS = (4 * HALO_N + 127) // 128


def exchange_halos_async(st, ctx):
    """Stage A + start of Stage B: pack, async D2H, sync, post Irecv/Isend.
    Returns the list of MPI requests."""
    comm = ctx["cart_comm"]
    sb = st.stream_boundary
    N = HALO_N

    # Stage A: pack on GPU, async D2H into pinned buffer
    pack_halos_kernel[HALO_BLOCKS, 128, sb](st.d_u, st.d_send, NY_LOC, NX_LOC, N)
    st.d_send.copy_to_host(st.h_send, stream=sb)
    sb.synchronize()   # host send buffers are ready

    hs, hr = st.h_send, st.h_recv
    reqs = []
    # receives (message moving UP arrives from the DOWN neighbor, etc.)
    reqs.append(comm.Irecv(hr[0:N],       source=ctx["nbr_up"],    tag=TAG_DOWN))
    reqs.append(comm.Irecv(hr[N:2*N],     source=ctx["nbr_down"],  tag=TAG_UP))
    reqs.append(comm.Irecv(hr[2*N:3*N],   source=ctx["nbr_left"],  tag=TAG_RIGHT))
    reqs.append(comm.Irecv(hr[3*N:4*N],   source=ctx["nbr_right"], tag=TAG_LEFT))
    # sends
    reqs.append(comm.Isend(hs[0:N],       dest=ctx["nbr_up"],      tag=TAG_UP))
    reqs.append(comm.Isend(hs[N:2*N],     dest=ctx["nbr_down"],    tag=TAG_DOWN))
    reqs.append(comm.Isend(hs[2*N:3*N],   dest=ctx["nbr_left"],    tag=TAG_LEFT))
    reqs.append(comm.Isend(hs[3*N:4*N],   dest=ctx["nbr_right"],   tag=TAG_RIGHT))
    return reqs


def complete_halo_exchange(st, reqs):
    """Stage C (first half): Waitall, async H2D, unpack into ghost cells."""
    MPI.Request.Waitall(reqs)
    sb = st.stream_boundary
    st.d_recv.copy_to_device(st.h_recv, stream=sb)
    unpack_halos_kernel[HALO_BLOCKS, 128, sb](st.d_u, st.d_recv, NY_LOC, NX_LOC, HALO_N)


# ==============================================================================
# TASK 4: 2D shared-memory tiled stencil + step orchestrators
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, un, kappa, S, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    smem = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    H = u.shape[0]
    W = u.shape[1]

    # center cell
    if i < H and j < W:
        smem[ty + 1, tx + 1] = u[i, j]
        # 1-cell halo apron (loaded by edge threads of the tile)
        if ty == 0 and i - 1 >= 0:
            smem[0, tx + 1] = u[i - 1, j]
        if ty == TILE_DIM - 1 and i + 1 < H:
            smem[SHARED_DIM - 1, tx + 1] = u[i + 1, j]
        if tx == 0 and j - 1 >= 0:
            smem[ty + 1, 0] = u[i, j - 1]
        if tx == TILE_DIM - 1 and j + 1 < W:
            smem[ty + 1, SHARED_DIM - 1] = u[i, j + 1]

    cuda.syncthreads()   # all threads reach this barrier (no early return above)

    if i <= r_end and j <= c_end:
        c = smem[ty + 1, tx + 1]
        n = smem[ty, tx + 1]
        s = smem[ty + 2, tx + 1]
        w = smem[ty + 1, tx]
        e = smem[ty + 1, tx + 2]
        k0 = kappa[i, j]
        kn = 0.5 * (kappa[i - 1, j] + k0)
        ks = 0.5 * (kappa[i + 1, j] + k0)
        kw = 0.5 * (kappa[i, j - 1] + k0)
        ke = 0.5 * (kappa[i, j + 1] + k0)
        diff = (ks * (s - c) - kn * (c - n)) * inv_dx2 \
             + (ke * (e - c) - kw * (c - w)) * inv_dy2
        un[i, j] = c + dt * (diff - lam * c * c * c + S[i, j])


def launch_stencil(st, stream, r0, r1, c0, c1):
    gy = (r1 - r0 + 1 + TILE_DIM - 1) // TILE_DIM
    gx = (c1 - c0 + 1 + TILE_DIM - 1) // TILE_DIM
    tiled_stencil_kernel[(gx, gy), (TILE_DIM, TILE_DIM), stream](
        st.d_u, st.d_un, st.d_kappa, st.d_S, r0, r1, c0, c1,
        st.dt, 1.0 / (DX * DX), 1.0 / (DY * DY), LAMBDA_DAMP)


def step_serialized(st, ctx):
    """Baseline: everything on one stream, no overlap."""
    sb = st.stream_boundary
    reqs = exchange_halos_async(st, ctx)
    complete_halo_exchange(st, reqs)
    launch_stencil(st, sb, 1, NY_LOC, 1, NX_LOC)       # whole owned domain
    sb.synchronize()
    st.d_u, st.d_un = st.d_un, st.d_u


def step_overlapped(st, ctx):
    """Dual-stream: interior compute overlaps with MPI exchange."""
    sb, si = st.stream_boundary, st.stream_interior
    ny, nx = NY_LOC, NX_LOC

    reqs = exchange_halos_async(st, ctx)                    # Stage A + B (posts)
    launch_stencil(st, si, 2, ny - 1, 2, nx - 1)           # deep interior, concurrent
    complete_halo_exchange(st, reqs)                        # Stage C (wait, H2D, unpack)
    launch_stencil(st, sb, 1, 1, 1, nx)                     # top ring row
    launch_stencil(st, sb, ny, ny, 1, nx)                   # bottom ring row
    launch_stencil(st, sb, 2, ny - 1, 1, 1)                 # left ring column
    launch_stencil(st, sb, 2, ny - 1, nx, nx)               # right ring column
    sb.synchronize()
    si.synchronize()
    st.d_u, st.d_un = st.d_un, st.d_u


if __name__ == "__main__":
    github_user = sys.argv[1]
    ctx = init_distributed_context(github_user)
    comm = ctx["cart_comm"]
    u0, kappa, S, kmax_local = initialize_subdomains(ctx)
    kmax = comm.allreduce(kmax_local, op=MPI.MAX)
    dt = 0.5 * DX ** 2 / (4.0 * kmax)

    TEST_STEPS = 20

    def run_mode(step_fn, nsteps):
        st = SolverState(u0, kappa, S, dt)
        comm.Barrier()
        t0 = time.perf_counter()
        for _ in range(nsteps):
            step_fn(st, ctx)
        cuda.synchronize()
        comm.Barrier()
        ms = (time.perf_counter() - t0) * 1e3
        return st.d_u.copy_to_host(), ms

    run_mode(step_serialized, 3)     # warm-up (JIT compile)
    run_mode(step_overlapped, 3)

    u_ser, ms_ser = run_mode(step_serialized, TEST_STEPS)
    u_ovl, ms_ovl = run_mode(step_overlapped, TEST_STEPS)

    local_diff = float(np.max(np.abs(u_ser - u_ovl)))
    max_diff = comm.allreduce(local_diff, op=MPI.MAX)

    dxy = DX * DY
    e_local = float(np.sum(u_ovl[1:NY_LOC + 1, 1:NX_LOC + 1]) * dxy)
    E = comm.allreduce(e_local, op=MPI.SUM)
    e0_local = float(np.sum(u0[1:NY_LOC + 1, 1:NX_LOC + 1]) * dxy)
    E0 = comm.allreduce(e0_local, op=MPI.SUM)

    print(f"rank={ctx['rank']} serialized={ms_ser:.2f} ms overlapped={ms_ovl:.2f} ms "
          f"u.max={u_ovl.max():.6f} finite={bool(np.isfinite(u_ovl).all())}")
    if ctx["rank"] == 0:
        print(f"parity max|ser-ovl| after {TEST_STEPS} steps: {max_diff:.3e}")
        print(f"E(step 0)={E0:.10f}  E(step {TEST_STEPS})={E:.10f}")

Overwriting distributed_cuda_solver.py


In [18]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

rank=0 serialized=64.82 ms overlapped=116.81 ms u.max=1.491240 finite=True
rank=1 serialized=65.35 ms overlapped=117.00 ms u.max=1.336764 finite=True
parity max|ser-ovl| after 20 steps: 0.000e+00
E(step 0)=0.1581981945  E(step 20)=0.1581847410
rank=2 serialized=64.97 ms overlapped=118.73 ms u.max=1.485057 finite=True
rank=3 serialized=64.82 ms overlapped=116.00 ms u.max=0.260055 finite=True



In [16]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python3 distributed_cuda_solver.py 230103010-wq 2>&1 | tail -n 60

Traceback (most recent call last):
Traceback (most recent call last):
  File "/content/distributed_cuda_solver.py", line 318, in <module>
    run_mode(step_serialized, 3)     # warm-up (JIT compile)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/content/distributed_cuda_solver.py", line 310, in run_mode
    t0 = time.perf_counter()
         ^^^^
NameError: name 'time' is not defined. Did you forget to import 'time'?
Traceback (most recent call last):
  File "/content/distributed_cuda_solver.py", line 318, in <module>
    run_mode(step_serialized, 3)     # warm-up (JIT compile)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/content/distributed_cuda_solver.py", line 310, in run_mode
    t0 = time.perf_counter()
         ^^^^
NameError: name 'time' is not defined. Did you forget to import 'time'?
Traceback (most recent call last):
  File "/content/distributed_cuda_solver.py", line 318, in <module>
    run_mode(step_serialized, 3)     # warm-up (JIT compile)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "